# Stage 1 — Exploratory Data Analysis
## Beijing Multi-Site Air Quality Dataset

**Goal:** Understand the data before touching any model.  
**Rule:** No transformations here — only observation and decisions.

**What we answer in this notebook:**
1. What does the data look like? (shape, types, samples)
2. How much is missing, and where?
3. What is the distribution of PM2.5 — our target?
4. Are there outliers that will hurt the model?
5. How do features correlate with each other and with PM2.5?
6. Are there time patterns (seasonal, hourly)?
7. Do stations behave differently?

**Book references:** Géron Ch. 2 (Look at the Big Picture) | Bruce Ch. 1 (EDA)

---
## 0. Setup

In [ ]:
import sys
sys.path.insert(0, '..')  # allow imports from src/

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from src.data.data_loader import load_processed_data

# Consistent plot style
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120
FIGURES_DIR = '../outputs/figures/'

print('Libraries loaded.')

---
## 1. Load Data

In [ ]:
df = load_processed_data()
print(f'Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
df.head(3)

In [ ]:
df.dtypes

In [ ]:
df.describe().round(2)

---
## 2. Missing Values

**Why this matters:** Sensors go offline. Missing values are not random — they cluster by time and station.  
Blindly dropping rows loses time-continuity. We need a deliberate strategy.

In [ ]:
total = len(df)
null_summary = (
    df.isnull().sum()
    .to_frame('null_count')
    .assign(pct=lambda x: (x['null_count'] / total * 100).round(2))
    .query('null_count > 0')
    .sort_values('pct', ascending=False)
)
print(f'Columns with nulls: {len(null_summary)}')
null_summary

In [ ]:
# Nulls broken down by station — are some sensors worse than others?
station_nulls = (
    df.groupby('station')['PM2.5']
    .apply(lambda x: x.isnull().sum())
    .rename('PM2.5_nulls')
    .reset_index()
    .assign(pct=lambda x: (x['PM2.5_nulls'] / df.groupby('station').size().values * 100).round(2))
    .sort_values('pct', ascending=False)
)

fig, ax = plt.subplots(figsize=(10, 4))
sns.barplot(data=station_nulls, x='station', y='pct', ax=ax, color='steelblue')
ax.set_title('PM2.5 Missing % by Station')
ax.set_xlabel('')
ax.set_ylabel('Missing (%)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(FIGURES_DIR + 'null_by_station.png')
plt.show()

**Decision to make after this cell:**  
- If any station has > 20% PM2.5 missing → flag it  
- Forward-fill within each station group (sensors fail briefly, then recover)  
- Rows where PM2.5 is still null after fill → drop (target cannot be imputed)

---
## 3. Target Variable — PM2.5 Distribution

**Why this matters:** Linear regression assumes errors are normally distributed.  
If PM2.5 is heavily skewed, predictions will be systematically worse in the high-pollution range — which is exactly where accuracy matters most for health.

In [ ]:
pm25 = df['PM2.5'].dropna()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Raw distribution
axes[0].hist(pm25, bins=100, color='steelblue', edgecolor='none')
axes[0].set_title('PM2.5 — Raw')
axes[0].set_xlabel('µg/m³')

# Log-transformed
log_pm25 = np.log1p(pm25)  # log1p handles zeros safely
axes[1].hist(log_pm25, bins=100, color='seagreen', edgecolor='none')
axes[1].set_title('PM2.5 — log1p transformed')
axes[1].set_xlabel('log(µg/m³ + 1)')

# Box plot — spotting extreme outliers
axes[2].boxplot(pm25, vert=True, patch_artist=True,
                boxprops=dict(facecolor='steelblue', alpha=0.6))
axes[2].set_title('PM2.5 — Box Plot')
axes[2].set_ylabel('µg/m³')

plt.suptitle('PM2.5 Distribution Analysis', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(FIGURES_DIR + 'pm25_distribution.png')
plt.show()

print(f'Skewness (raw):         {pm25.skew():.2f}')
print(f'Skewness (log1p):       {log_pm25.skew():.2f}')
print(f'Max PM2.5:              {pm25.max():.0f} µg/m³')
print(f'WHO daily limit:        15 µg/m³')
print(f'Rows above 300 µg/m³:   {(pm25 > 300).sum():,} ({(pm25>300).mean()*100:.1f}%)')

**Decision to make after this cell:**  
- Skewness > 1.5 → strong case for log1p transform before modelling  
- If we transform, remember to inverse-transform predictions back to µg/m³ for interpretation  
- Extreme values (> 500 µg/m³) — are they real or sensor errors? Check against station neighbours

---
## 4. Outlier Analysis

**Why this matters:** Extreme values inflate RMSE and distort coefficient estimates in linear models.  
But in pollution data, extreme events are real — we don't drop them blindly.

In [ ]:
NUMERIC_COLS = ['PM2.5', 'PM10', 'SO2', 'NO2', 'CO', 'O3', 'TEMP', 'PRES', 'DEWP', 'RAIN', 'WSPM']

# IQR-based outlier count — not dropping, just counting
outlier_report = []
for col in NUMERIC_COLS:
    s = df[col].dropna()
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    n_out = ((s < q1 - 1.5 * iqr) | (s > q3 + 1.5 * iqr)).sum()
    outlier_report.append({'feature': col, 'outliers': n_out, 'pct': round(n_out / len(s) * 100, 2)})

pd.DataFrame(outlier_report).sort_values('pct', ascending=False)

---
## 5. Feature Distributions

**Why this matters:** Features on very different scales cause problems for Ridge, Lasso, and ElasticNet — they are distance-sensitive.  
Seeing distributions now tells us what scaling strategy to use in Stage 3.

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(16, 10))
axes = axes.flatten()

for i, col in enumerate(NUMERIC_COLS):
    axes[i].hist(df[col].dropna(), bins=60, color='steelblue', edgecolor='none')
    axes[i].set_title(col, fontsize=10)
    axes[i].set_xlabel('')

# Hide the unused subplot
for j in range(len(NUMERIC_COLS), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Feature Distributions', fontsize=13)
plt.tight_layout()
plt.savefig(FIGURES_DIR + 'feature_distributions.png')
plt.show()

---
## 6. Correlation Analysis

**Why this matters:**  
- High correlation between **features** = multicollinearity → Ridge/Lasso needed  
- High correlation between a **feature and PM2.5** = that feature will be important  
- Knowing this now guides feature engineering decisions in Stage 4

In [ ]:
corr = df[NUMERIC_COLS].corr()

fig, ax = plt.subplots(figsize=(11, 9))
mask = np.triu(np.ones_like(corr, dtype=bool))  # show lower triangle only
sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
    center=0, linewidths=0.4, ax=ax, annot_kws={'size': 8}
)
ax.set_title('Feature Correlation Matrix', fontsize=13)
plt.tight_layout()
plt.savefig(FIGURES_DIR + 'correlation_heatmap.png')
plt.show()

In [ ]:
# Which features correlate most strongly with PM2.5?
pm25_corr = (
    corr['PM2.5']
    .drop('PM2.5')
    .sort_values(key=abs, ascending=False)
)

fig, ax = plt.subplots(figsize=(8, 5))
colors = ['tomato' if v < 0 else 'steelblue' for v in pm25_corr]
pm25_corr.plot(kind='bar', ax=ax, color=colors, edgecolor='none')
ax.axhline(0, color='black', linewidth=0.8)
ax.set_title('Correlation with PM2.5')
ax.set_ylabel('Pearson r')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(FIGURES_DIR + 'pm25_correlations.png')
plt.show()

pm25_corr

---
## 7. Time Patterns

**Why this matters:** PM2.5 is not random across time — rush hours, seasons, and weather cycles all matter.  
If the model doesn't know about time, it will produce errors that are structured, not random — a sign of a missing feature.

In [ ]:
# Hourly pattern — does pollution peak at rush hour?
hourly = df.groupby('hour')['PM2.5'].median()

fig, ax = plt.subplots(figsize=(10, 4))
hourly.plot(ax=ax, marker='o', color='steelblue', linewidth=2)
ax.set_title('Median PM2.5 by Hour of Day')
ax.set_xlabel('Hour')
ax.set_ylabel('PM2.5 (µg/m³)')
ax.set_xticks(range(0, 24))
plt.tight_layout()
plt.savefig(FIGURES_DIR + 'pm25_by_hour.png')
plt.show()

In [ ]:
# Monthly pattern — does winter have worse pollution?
monthly = df.groupby('month')['PM2.5'].median()

fig, ax = plt.subplots(figsize=(10, 4))
monthly.plot(kind='bar', ax=ax, color='steelblue', edgecolor='none')
ax.set_title('Median PM2.5 by Month')
ax.set_xlabel('Month')
ax.set_ylabel('PM2.5 (µg/m³)')
ax.set_xticklabels(['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec'], rotation=45)
plt.tight_layout()
plt.savefig(FIGURES_DIR + 'pm25_by_month.png')
plt.show()

In [ ]:
# Yearly trend — is pollution getting better or worse over 2013-2017?
yearly = df.groupby('year')['PM2.5'].median()

fig, ax = plt.subplots(figsize=(7, 4))
yearly.plot(marker='o', ax=ax, color='tomato', linewidth=2)
ax.set_title('Median PM2.5 by Year')
ax.set_xlabel('Year')
ax.set_ylabel('PM2.5 (µg/m³)')
plt.tight_layout()
plt.savefig(FIGURES_DIR + 'pm25_by_year.png')
plt.show()

---
## 8. Station Comparison

**Why this matters:** Geography is real signal. A station near a highway will behave differently than one in a park.  
If stations differ significantly, the `station` column must be encoded as a feature — not ignored.

In [ ]:
station_stats = (
    df.groupby('station')['PM2.5']
    .agg(['median', 'mean', 'std', 'max'])
    .round(1)
    .sort_values('median', ascending=False)
)
station_stats

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
df.boxplot(
    column='PM2.5', by='station', ax=ax,
    flierprops=dict(marker='.', markersize=1, alpha=0.3),
    patch_artist=False
)
ax.set_title('PM2.5 Distribution by Station')
ax.set_xlabel('')
ax.set_ylabel('PM2.5 (µg/m³)')
plt.suptitle('')  # remove default boxplot title
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(FIGURES_DIR + 'pm25_by_station.png')
plt.show()

---
## 9. Wind Direction

**Why this matters:** Wind direction is categorical but circular — North is close to NNW, not far from it.  
One-hot encoding treats all directions as equally distant, which is wrong.  
We'll encode it as sin/cos in Stage 4.

In [ ]:
# How many unique wind directions? What's the distribution?
wd_counts = df['wd'].value_counts()
print(f'Unique wind directions: {df["wd"].nunique()}')
print(wd_counts.head(10))

# Median PM2.5 by wind direction — does wind from certain directions bring more pollution?
wd_pm25 = (
    df.groupby('wd')['PM2.5']
    .median()
    .sort_values(ascending=False)
)

fig, ax = plt.subplots(figsize=(12, 4))
wd_pm25.plot(kind='bar', ax=ax, color='steelblue', edgecolor='none')
ax.set_title('Median PM2.5 by Wind Direction')
ax.set_xlabel('Wind Direction')
ax.set_ylabel('PM2.5 (µg/m³)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig(FIGURES_DIR + 'pm25_by_wind.png')
plt.show()

---
## 10. EDA Summary — Decisions for Stage 3 Onwards

Fill in after running all cells above. This becomes your contract for preprocessing.

| Decision | Finding | Action |
|----------|---------|--------|
| Null strategy | [fill after cell 2] | Forward-fill by station → drop remaining PM2.5 nulls |
| PM2.5 skewness | [fill after cell 3] | Log1p transform if skewness > 1.5 |
| Outliers | [fill after cell 4] | Flag extreme values; do not blindly drop |
| Feature scaling | [fill after cell 5] | StandardScaler — all features on different scales |
| Multicollinearity | [fill after cell 6] | Ridge/Lasso/ElasticNet required |
| Time features | [fill after cell 7] | Add `hour`, `month` as features in Stage 4 |
| Station | [fill after cell 8] | Encode `station` — significant variation observed |
| Wind direction | [fill after cell 9] | Sin/cos cyclical encoding in Stage 4 |